# Impedance ML Engineering — Session 17
## Quantify model uncertainty and inspect shift
**Core: 60 minutes · ML evaluation · Optional extension: 20–40 minutes**

## Why this matters
How stable is a model advantage, and how might measurement or population shift break it?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Bootstrap patients rather than individual pairs.
- Separate mean-performance uncertainty from predictive intervals.
- Inspect site/protocol errors with counts.
- Stress-test preprocessing and controlled input shifts.

## R-to-Python / prerequisite bridge
Cluster bootstrap → sample patient IDs with replacement and retain duplicate draws; sampling IDs with `.isin` alone drops bootstrap multiplicity.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
A paired comparison uses each patient as their own model-comparison unit, improving alignment of the performance difference. Resampling patient-level mean errors estimates uncertainty for patient-average performance; pooling visit errors answers a different weighting question.

Input drift is a distribution change in predictors; performance drift is a change in error against later observed outcomes. An input perturbation probes sensitivity under a chosen scenario and does not validate that scenario physically. A conditional performance interval omits model-selection/training uncertainty unless the entire selection/fitting process is included in resampling.

Use `np.random.default_rng(42).choice(...)` with replacement and retain multiplicity. Report subgroup counts alongside errors. Do not transfer iid-row conformal guarantees to dependent pairs without a valid group-aware argument.

## Task 1 — Freeze a comparison on validation (5 min)
Train a fixed summary ridge using training only; compare persistence on validation. Build aligned row-level absolute errors, IDs, and horizons. Declare patient-average MAE difference as the statistic; negative model-minus-baseline favors ridge. Predict the patient-level table’s row count (18).

## Task 2 — Bootstrap independent units (10 min)
Average each model’s absolute error within patient, calculate paired patient error differences, and sample 18 patient-level differences with replacement 1000 times using seed 42. Report percentile 95% interval. This preserves within-patient dependence for the chosen patient-weighted statistic. Explain why pooling all sampled pairs gives a different weighting.

## Task 3 — Interpret the uncertainty scope (5 min)
State that this interval is conditional on fitted models, selected recipe, and observed validation population. It does not include all training/model-selection uncertainty. Distinguish an interval around mean MAE difference from a predictive interval for one future visit.

## Task 4 — Inspect subgroup coverage (7 min)
Report validation patient counts, pair counts, and patient-average errors by current protocol/site/horizon. Patients can contribute to more than one protocol/horizon subgroup. Inspect sparse cells and explain why ranking subgroups without uncertainty is weak evidence.

## Task 5 — Run a site-held-out development experiment (10 min)
Within all 72 development patients, hold out site_3 patients and train on sites 1/2. Fit a fresh fixed ridge; report held-site MAE and persistence. Ensure site is constant within patient and no patient overlap. Label this a different generalization question; never include the reserved 24 test patients in site selection.

## Task 6 — Stress-test an explicit protocol offset (8 min)
On a copy of validation inputs, add 0.2 kΩ to current diagonal mean as a hypothetical measurement shift; keep outcomes fixed only for this sensitivity experiment. Compare prediction/error changes. Explain this is an invented input perturbation, not a validated model of a protocol change; matrix and summary changes would need consistent reconstruction in a real stress test.

## Task 7 — Test schema and missingness failures (6 min)
Set one validation age missing, introduce an unseen site category, and remove a required column in separate copies. Document what the pipeline handles and what should fail clearly. Keep baseline predictions unchanged and do not mutate the original validation table.

## Task 8 — Write a monitoring proposal (1 min)
Specify one schema check, one input-distribution summary, and one delayed-outcome performance measure with minimum sample counts. Explain that input drift alone does not prove worse performance.

## Retrieval practice
Explain why sampling patients then filtering with `.isin(sampled_ids)` loses repeated bootstrap draws, and give a correct representation.

## Professional reasoning
1. Why can a narrow conditional interval coexist with unreliable deployment performance?
2. What could go wrong when site is both a measurement setting and a population marker?
3. How would a split-conformal prediction-interval exercise need to address dependent visit pairs?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Compare uncertainty on patient-weighted and pair-weighted MAE using an explicit repeated cluster draw. Discuss why conformal guarantees from independent rows do not transfer automatically to these pairs.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)